# Sprint 53 · Task B · Step B5 — combined prompt C (Stage 1b) + held-out confirmation (Stage 2)

**Every session:** Runtime → Change runtime type → **T4 GPU**, then run cells 1 → 5.

| Cell | What it does | When |
|---|---|---|
| 1 | GPU check | every session |
| 2 | Upload zip(s) and unpack | every session |
| 3 | Install + start Ollama, download both judges, check the new prompts | every session (~3–6 min) |
| 4 | Progress of Stage 1b and Stage 2 | every session |
| 5 | Define `save_checkpoint()` | every session |
| 6 | **Part 1a** — smoke test of the 10 new prompts (20 calls) + time estimates | once |
| 7 | **Part 1b** — Stage 1b: prompt C on the development half (436 calls, ~20 min) + analysis → decides whether C joins Stage 2 | once, right after cell 6 |
| 8 | **Part 2** — Stage 2 held-out confirmation, 30-min chunks with checkpoint downloads | ⚠️ only after Claude confirms the cell 6–7 output |
| 9 | Manual checkpoint | before closing the tab |


In [ ]:
# 1. GPU check - should show a Tesla T4 with ~15 GB
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


**Cell 2 — what to upload** (one file at a time; the cell keeps asking until it has everything)
* **First time:** your newest Stage 1 checkpoint (`sprint53_taskB_run_20261003_0419.zip`, or `sprint53_taskB_B4_<time>.zip` if you ran the optional analysis notebook), then `sprint53_taskB_step5_code.zip`.
* **Later sessions:** only the newest `sprint53_taskB_run_<time>.zip` checkpoint (it already contains the Step B5 code).


In [ ]:
# 2. Upload and unpack
import os, zipfile, glob
from google.colab import files
os.chdir('/content')

def unpack(path):
    with zipfile.ZipFile(path) as z:
        tops = {p.split('/')[0] for p in z.namelist() if p.strip('/')}
        if tops != {'sprint53'}:
            raise SystemExit(f'!! {path} does not contain a single sprint53/ folder (found {tops}).')
        z.extractall('/content')
    print('   unpacked', path)

have_data = lambda: os.path.exists('/content/sprint53/results_taskB/B1_repeat_T07.jsonl')
have_code = lambda: os.path.exists('/content/sprint53/taskB/check_prompts_b5.py')
code_zips = []
while not (have_data() and have_code()):
    need = []
    if not have_data(): need.append('newest Stage 1 / run checkpoint (sprint53_taskB_run_<time>.zip)')
    if not have_code(): need.append('Step B5 code (sprint53_taskB_step5_code.zip)')
    print('Please upload ONE file now:', ' / '.join(need))
    for name in files.upload():
        unpack(name)
        if 'step5_code' in name.lower(): code_zips.append(name)
        else:
            for c in code_zips: unpack(c)
os.chdir('/content/sprint53')
print('\nReady. Task B result files:', len(glob.glob('results_taskB/*.jsonl')))


In [ ]:
# 3. Install + start Ollama, download both judges, record versions, check the new prompts
%cd /content/sprint53
!bash colab/setup_ollama.sh
!python taskB/run_taskb.py --check
!python taskB/check_prompts_b5.py


In [ ]:
# 4. Progress (first session: Stage 1b 2 phases, TOTAL 436, 0 done; Stage 2 needs Stage 1b first)
!python taskB/run_taskb.py --status --stage 1b
if os.path.exists('data/taskB/stage2_finalists.json'):
    !cat data/taskB/stage2_finalists.json
    !python taskB/run_taskb.py --status --stage 2


In [ ]:
# 5. Checkpoint helper + run loop
import datetime, zipfile, os, time, json
from google.colab import files
def save_checkpoint():
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M')
    out = f'/content/sprint53_taskB_run_{stamp}.zip'
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for root, dirs, fnames in os.walk('/content/sprint53'):
            dirs[:] = [d for d in dirs if d not in ('node_modules', '.venv', '__pycache__')]
            for f in fnames:
                if f.endswith('.zip'): continue
                full = os.path.join(root, f); z.write(full, os.path.relpath(full, '/content'))
    print(f'Checkpoint: {out} ({os.path.getsize(out)/1e6:.1f} MB) - downloading...')
    files.download(out)

def run_stage(stage, session_budget_min=195, chunk_min=30):
    start = time.time()
    while True:
        left = session_budget_min - (time.time() - start) / 60
        if left < 3:
            print('Session budget used - keep the last checkpoint and continue next session.'); return False
        get_ipython().system(f'python taskB/run_taskb.py --run --stage {stage} --budget {int(min(chunk_min, left))}m')
        rc = get_ipython().user_ns.get('_exit_code', 0)
        save_checkpoint()
        if rc != 0:
            print('!! The runner stopped with an error (see above). Re-run cell 3, then this cell.'); return False
        if json.load(open(f'results_taskB/progress_stage{stage}.json'))['next_phase'] is None:
            print(f'STAGE {stage} COMPLETE'); return True
print('save_checkpoint() and run_stage() are ready')


In [ ]:
# 6. Part 1a - smoke test of the 10 new prompts (20 calls) + time estimates
!python taskB/run_taskb.py --smoke --prompts b5


In [ ]:
# 7. Part 1b - Stage 1b: combined prompt C on the development half, then the analysis that decides
#    whether C joins Stage 2. Refuses to start if the smoke test had an unreadable or cut-off answer.
sm = [json.loads(l) for l in open('results_taskB/smoke_b5.jsonl')]
bad = [r['prompt_id'] + '/' + r['model'] for r in sm if r['parse_status'] == 'malformed' or r['done_reason'] == 'length']
if len(sm) != 20 or bad:
    print('!! Smoke test not clean:', bad or f'{len(sm)} calls', '- send the cell 6 output to Claude; do not continue.')
elif run_stage('1b'):
    !python taskB/analyze_stage1.py | sed -n '/Stage 1 decision rule/,$p'
    save_checkpoint()


⚠️ **Stop here.** Send Claude the outputs of cells 6 and 7. Run cell 8 only after Claude confirms the finalists.

In [ ]:
# 8. Part 2 - Stage 2 on the held-out half (about 2.5-3.5 h; likely 2 sessions).
#    Next session: cells 1-5 (upload only the newest checkpoint), then this cell again - it continues.
!cat data/taskB/stage2_finalists.json
run_stage('2')


In [ ]:
# 9. Manual checkpoint (run before closing the tab)
save_checkpoint()
